# 12 - Rhythm-context features, model selection, and freezing the protocol

**The rules in sections 1-6 were written and committed before any model was run with the new features**, so the order of events is visible in the git history: the first commit of this notebook contains the rules and the code but no results.

Three things are settled here, in this order, by rules stated in advance: (1) whether to adopt two new rhythm-context features; (2) which model goes to the test set; (3) what is frozen. The test set stays sealed throughout - the 5 test patients are dropped from memory below.

## 1. Why

The evidence is from notebooks 09-11, all development data. Each of the three models flags atrial fibrillation and flutter heavily (for the logistic regression, those two rhythms are 12% of Normal beats but 59% of all false alarms), and all three miss atrial beats (8-10% flagged). Both problems have the same root: a beat and its two neighbouring intervals cannot tell *one premature beat in a regular rhythm* from *an irregular rhythm*, and absolute intervals are confounded by each patient's resting heart rate. That is a feature gap, not a model gap - which is why this comes before the final model choice.

## 2. The features (defined before the experiment)

Both use the 8 RR intervals that precede the beat's own pre-interval (the intervals ending at the previous 8 beats), each clipped to fixed physiological bounds (0.3-2.0 s):

- **`rr_hist_cv`** - coefficient of variation (standard deviation / mean) of those 8 intervals. Expected to be high in an irregular rhythm and low in a regular one, whatever the heart rate.
- **`rr_pre_vs_hist`** - the beat's own pre-interval divided by the mean of those 8. Expected to be well below 1 for a premature beat *relative to the patient's current rhythm*, near 1 in any regular rhythm (including a fast one, such as flutter).

No labels, nothing fitted, only RR intervals of the same record. At least 4 previous intervals are needed; the first few beats of each record get that record's own median. The 8-interval window was fixed in advance; no other window will be tried.

A descriptive check further down (the features behave as intended, and a hand recomputation of one beat matches) was done before this commit; no definition was changed after looking at it.

## 3. The experiment

Each of the three models, untuned and exactly as in notebooks 09-11, on the 13 baseline features versus the 15 features (13 + these 2). Same 12 development patients, same 4 grouped folds. This is a single change: no feature is removed, nothing else is altered.

## 4. Adoption rule

Adopt the two features - for **all** models, so the comparison stays fair - only if **both** hold:

- **(a)** pooled cross-validated PR-AUC is higher with the 15 features for at least 2 of the 3 models; and
- **(b)** for every model satisfying (a), the false-alarm rate on Normal beats in atrial fibrillation/flutter stretches is lower, and the recall of ventricular (`V`) beats does not fall by more than 2 percentage points.

Otherwise the 13 features stay. **One attempt:** no second feature idea, no other window, no variants.

## 5. Model selection rule

Using the chosen feature set: take the model with the highest pooled cross-validated PR-AUC. If its advantage over the logistic regression has a patient-level bootstrap 95% interval that includes zero, **select the logistic regression** instead (the simplest model is preferred whenever a difference is within noise).

## 6. What is frozen afterwards

Threshold **0.5** (the default; unvalidated, so the test evaluation will also report threshold-free metrics), hyperparameters exactly as in notebooks 09-11, the split exactly as in `results/metrics/record_split.json`, and the feature set decided in section 4. These are written to `results/metrics/frozen_protocol.json`. After that, any change to any of them means the test set is no longer a test.

In [ ]:
import sys
sys.path.append("..")

import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

from src.evaluation import annotated_rhythm, out_of_fold_probabilities, patient_bootstrap, summarize
from src.feature_extraction import (MODEL_FEATURES, MODEL_FEATURES_RHYTHM, RHYTHM_CONTEXT_FEATURES,
                                    add_record_relative_features, add_rhythm_context_features, load_beat_table)
from src.models import DEFAULT_CLIP_BOUNDS, make_gradient_boosting, make_logistic_regression, make_random_forest
from src.splitting import DEFAULT_SPLIT_PATH, apply_split, load_split

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

table = add_record_relative_features(load_beat_table()).sort_values(["record", "r_peak_sample"]).reset_index(drop=True)
table = add_rhythm_context_features(table)
data = apply_split(table, load_split())
dev = data[data.split == "train"].copy()
del table, data                                   # the test rows are gone from this notebook
dev["rhythm"] = annotated_rhythm(dev)             # analysis only - never a model input
print(f"development set: {len(dev)} beats, {dev.record.nunique()} patients; baseline features {len(MODEL_FEATURES)}, "
      f"with rhythm context {len(MODEL_FEATURES_RHYTHM)}")
print("missing values in the new features:", int(dev[RHYTHM_CONTEXT_FEATURES].isna().sum().sum()))

## Descriptive check: do the features behave as intended?

Median of each feature by annotated rhythm (Normal beats) and by beat type. Rhythm labels are used here for analysis only.

In [ ]:
print("rr_hist_cv - Normal beats by annotated rhythm")
print(dev[dev.label == "Normal"].groupby("rhythm").rr_hist_cv.agg(beats="size", median="median").round(3).to_string())
print()
print("rr_pre_vs_hist - by beat type")
print(dev.groupby("symbol", observed=True).rr_pre_vs_hist.agg(beats="size", median="median").round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

normal = dev[dev.label == "Normal"]
rhythms = ["(N", "(AFIB", "(AFL", "(B", "(SBR"]
axes[0].boxplot([normal.loc[normal.rhythm == r, "rr_hist_cv"] for r in rhythms], tick_labels=rhythms, showfliers=False)
axes[0].set_ylabel("rr_hist_cv (variability of the preceding intervals)")
axes[0].set_title("Normal beats: variability of the rhythm before the beat")

symbols = ["N", "L", "R", "A", "V", "F", "J"]
axes[1].boxplot([dev.loc[dev.symbol.astype(str) == s, "rr_pre_vs_hist"] for s in symbols], tick_labels=symbols, showfliers=False)
axes[1].axhline(1, color="grey", linestyle="--")
axes[1].set_ylabel("rr_pre_vs_hist (own interval / recent average)")
axes[1].set_title("By beat type: how early does the beat arrive?")

fig.tight_layout()
fig.savefig("../results/figures/24_rhythm_context_features.png", dpi=120)
plt.show()

## The experiment (section 3)

Out-of-fold probabilities for every model x feature set, from the same grouped cross-validation as before.

In [ ]:
models = {"logistic regression": make_logistic_regression, "random forest": make_random_forest,
          "gradient boosting": make_gradient_boosting}
feature_sets = {"13 baseline": MODEL_FEATURES, "15 with rhythm context": MODEL_FEATURES_RHYTHM}

oof = {}
for model_name, factory in models.items():
    for set_name, features in feature_sets.items():
        oof[(model_name, set_name)] = out_of_fold_probabilities(factory, dev, features=features)

pooled = pd.DataFrame({key: summarize(dev.is_abnormal, proba) for key, proba in oof.items()}).T
pooled = pooled[["precision", "recall", "f1", "false_alarm_rate", "roc_auc", "pr_auc"]].astype(float).round(3)
pooled.index.names = ["model", "features"]
pooled

In [ ]:
rows = []
for model_name in models:
    boot = patient_bootstrap(dev, oof[(model_name, "13 baseline")], oof[(model_name, "15 with rhythm context")], n_boot=1000, seed=0)
    for metric, label in [("pr_auc", "PR-AUC"), ("roc_auc", "ROC-AUC"), ("f1", "F1 at 0.5")]:
        diff = boot[metric + "_b"] - boot[metric + "_a"]
        rows.append((model_name, label, diff.mean(), *np.percentile(diff, [2.5, 97.5]), (diff > 0).mean()))
pd.DataFrame(rows, columns=["model", "metric (15 features minus 13)", "mean difference", "2.5th percentile",
                            "97.5th percentile", "share of resamples where 15 is better"]).round(3)

### The targeted metrics named in the adoption rule

Recall by beat type, and the false-alarm rate on Normal beats by rhythm (share of Normal beats flagged as abnormal, %).

In [ ]:
def targeted(proba):
    flagged = proba >= 0.5
    out = {f"recall {s}": 100 * flagged[dev.symbol.astype(str) == s].mean() for s in ["V", "A", "F"]}
    normal_beats = dev.label == "Normal"
    afib_afl = normal_beats & dev.rhythm.isin(["(AFIB", "(AFL"])
    out["false alarms AFIB+AFL"] = 100 * flagged[afib_afl].mean()
    for rhythm, name in [("(N", "sinus"), ("(AFIB", "AFIB"), ("(AFL", "AFL"), ("(SBR", "SBR"), ("(B", "bigeminy")]:
        out[f"false alarms {name}"] = 100 * flagged[normal_beats & (dev.rhythm == rhythm)].mean()
    return out


targets = pd.DataFrame({key: targeted(proba) for key, proba in oof.items()}).T.round(1)
targets.index.names = ["model", "features"]
targets

In [ ]:
per_patient = pd.DataFrame({
    key: [average_precision_score(part.is_abnormal, proba.loc[part.index]) for _, part in dev.groupby("record")]
    for key, proba in oof.items()
}, index=sorted(dev.record.unique()))
change = pd.DataFrame({model_name: per_patient[(model_name, "15 with rhythm context")] - per_patient[(model_name, "13 baseline")]
                       for model_name in models}).round(3)
abnormal_pct = dev.groupby("record").is_abnormal.mean().mul(100).round(0).astype(int).rename("% abnormal")
print("Change in per-patient PR-AUC from adding the two features (15 minus 13)")
pd.concat([abnormal_pct, change], axis=1)

## Applying the adoption rule (section 4), exactly as written

In [ ]:
rule_a, rule_b = {}, {}
for model_name in models:
    base, new = oof[(model_name, "13 baseline")], oof[(model_name, "15 with rhythm context")]
    rule_a[model_name] = summarize(dev.is_abnormal, new)["pr_auc"] > summarize(dev.is_abnormal, base)["pr_auc"]
    t_base, t_new = targeted(base), targeted(new)
    rule_b[model_name] = (t_new["false alarms AFIB+AFL"] < t_base["false alarms AFIB+AFL"]
                          and t_new["recall V"] >= t_base["recall V"] - 2.0)

satisfying_a = [m for m in models if rule_a[m]]
adopt = len(satisfying_a) >= 2 and all(rule_b[m] for m in satisfying_a)

print("(a) PR-AUC higher with the 15 features:", rule_a)
print("(b) fewer AFIB/AFL false alarms and V recall not down by more than 2 points:", rule_b)
print(f"models satisfying (a): {satisfying_a}")
print()
print("DECISION:", "ADOPT the two rhythm-context features for all models" if adopt else "KEEP the 13 baseline features")
chosen_set = "15 with rhythm context" if adopt else "13 baseline"

## Applying the model selection rule (section 5), exactly as written

With the chosen feature set, take the highest pooled PR-AUC; compare it with the logistic regression by patient-level bootstrap (2,000 resamples).

In [ ]:
scores = {m: summarize(dev.is_abnormal, oof[(m, chosen_set)])["pr_auc"] for m in models}
best = max(scores, key=scores.get)
print(f"feature set: {chosen_set}")
print("pooled cross-validated PR-AUC:", {m: round(s, 3) for m, s in scores.items()})
print("highest:", best)

if best == "logistic regression":
    selected = "logistic regression"
    print("The highest is the logistic regression itself -> selected.")
else:
    boot = patient_bootstrap(dev, oof[("logistic regression", chosen_set)], oof[(best, chosen_set)], n_boot=2000, seed=0)
    diff = boot["pr_auc_b"] - boot["pr_auc_a"]
    low, high = np.percentile(diff, [2.5, 97.5])
    print(f"{best} minus logistic regression, PR-AUC: mean {diff.mean():.3f}, 95% interval [{low:.3f}, {high:.3f}]")
    selected = best if low > 0 else "logistic regression"
    print("Interval excludes zero -> select", best if low > 0 else "the interval includes zero -> select the logistic regression (simplest within noise)")

print()
print("SELECTED MODEL:", selected)

## Freezing the protocol (section 6)

Everything the test evaluation will use, written to a file that is committed **before** the test set is looked at.

In [ ]:
SETTINGS = {
    "logistic regression": {"factory": "src.models.make_logistic_regression", "C": 1.0, "penalty": "l2", "class_weight": None,
                            "max_iter": 1000, "scaling": "StandardScaler fitted on the training data only",
                            "clipping (fixed bounds)": {k: list(v) for k, v in DEFAULT_CLIP_BOUNDS.items()}},
    "random forest": {"factory": "src.models.make_random_forest", "n_estimators": 300, "class_weight": None, "random_state": 42},
    "gradient boosting": {"factory": "src.models.make_gradient_boosting", "max_iter": 100, "class_weight": None,
                          "early_stopping": False, "random_state": 42},
}
final_features = MODEL_FEATURES_RHYTHM if adopt else MODEL_FEATURES
selected_dev = summarize(dev.is_abnormal, oof[(selected, chosen_set)])

frozen = {
    "selected_model": selected,
    "model_settings": SETTINGS[selected],
    "feature_set": chosen_set,
    "features": final_features,
    "threshold": 0.5,
    "adoption_rule_outcome": {"adopt_rhythm_context_features": bool(adopt), "rule_a": {k: bool(v) for k, v in rule_a.items()},
                              "rule_b": {k: bool(v) for k, v in rule_b.items()}},
    "development_cross_validation_of_selected_model": {k: round(float(v), 4) if isinstance(v, float) else int(v)
                                                       for k, v in selected_dev.items()},
    "split_file_sha256": hashlib.sha256(Path(DEFAULT_SPLIT_PATH).read_bytes()).hexdigest(),
    "rules": {
        "selection": "highest pooled dev-CV PR-AUC; the logistic regression wins whenever the advantage over it has a "
                     "patient-level bootstrap 95% interval that includes zero",
        "test_set": "evaluated once, after this file is committed; nothing below may change afterwards",
    },
}
out_path = Path("../results/metrics/frozen_protocol.json")
out_path.write_text(json.dumps(frozen, indent=2) + "\n", encoding="utf-8")
print(json.dumps({k: frozen[k] for k in ["selected_model", "feature_set", "threshold"]}, indent=2))
print("written to", out_path)